# Insurance Claim Cost Prediction and Fraud Detection
### End-to-End Data Science Analysis

**Author:** Marzieh Abbasi  
**Course:** DSC 530 – Data Exploration and Analysis  
**Institution:** Bellevue University  

---

# Executive Summary

Insurance companies rely heavily on predictive analytics to estimate potential claim costs and detect fraudulent claims. Inaccurate predictions can result in financial loss, improper pricing strategies, and increased operational risk.

This project performs a complete end-to-end data science workflow on an insurance claims dataset. The analysis combines statistical analysis, exploratory data analysis (EDA), supervised machine learning, and unsupervised learning techniques.

The main goals of this project are:

• Identify key drivers of insurance claim costs  
• Develop predictive models for estimating claim amounts  
• Detect potentially fraudulent claims using classification algorithms  
• Discover hidden behavioral segments among policyholders  

The methodology implemented in this project reflects a real-world data science pipeline used in insurance analytics.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set(style="whitegrid")

print("Libraries successfully loaded.")

# Dataset Description

The dataset contains insurance claim records including customer demographics, policy information, and claim outcomes.

The dataset consists of approximately **1000 observations** with multiple features describing policyholder characteristics and incident details.

## Key Variables

| Variable | Description |
|--------|-------------|
| total_claim_amount | Total payout amount for the claim |
| age | Policyholder age |
| months_as_customer | Customer tenure |
| incident_severity | Severity level of the claim |
| fraud_reported | Binary indicator of whether fraud was reported |

Two main modeling tasks are explored in this analysis:

1. **Regression:** Predicting claim cost  
2. **Classification:** Detecting fraudulent claims

In [ ]:
df = pd.read_csv("../data/raw/insurance_claims.csv")

print("Dataset shape:", df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

# Data Cleaning

Real-world datasets frequently contain missing values and inconsistent data formatting.

In this dataset, missing values are represented using the placeholder symbol `"?"`. These values must be converted to proper missing indicators (`NaN`) to allow correct preprocessing.

Missing values are handled using:

• **Median imputation** for numerical variables  
• **Mode imputation** for categorical variables  

Median imputation is preferred for financial variables because it is more robust to outliers than the mean.

In [ ]:
df = df.replace("?", np.nan)

df = df.drop(columns=["_c39"], errors="ignore")

numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.head()

# Feature Engineering

Feature engineering is one of the most important steps in the machine learning pipeline. Creating informative features can significantly improve model performance.

In this analysis we create additional derived variables:

• **claim_per_month** – claim cost normalized by customer tenure  
• **age_group** – categorical grouping of policyholder age

These engineered variables can capture patterns not directly observable in the raw dataset.

In [ ]:
df["claim_per_month"] = df["total_claim_amount"] / (df["months_as_customer"] + 1)

df["age_group"] = pd.cut(
    df["age"],
    bins=[18,30,40,50,60,100],
    labels=["18-30","30-40","40-50","50-60","60+"]
)

In [ ]:
le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# Exploratory Data Analysis

Exploratory Data Analysis (EDA) helps uncover patterns and relationships within the dataset before building predictive models.

Key objectives of EDA:

• Understand the distribution of claim costs  
• Detect outliers  
• Identify relationships between variables  
• Explore potential predictive features

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df["total_claim_amount"],
    bins=30,
    kde=True
)

plt.title("Distribution of Insurance Claim Amounts")
plt.savefig("../reports/figures/claim_distribution.png")
plt.xlabel("Claim Amount")
plt.ylabel("Frequency")

plt.show()

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(12,8))

sns.heatmap(
    corr,
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.savefig("../reports/figures/correlation_matrix.png")
plt.show()

# Multicollinearity Analysis

Multicollinearity occurs when predictor variables are highly correlated with one another.

This can destabilize regression coefficients and reduce model interpretability.

To assess multicollinearity, we compute the **Variance Inflation Factor (VIF)** for each predictor variable.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).copy()

X_vif = numeric_df.drop(columns=["total_claim_amount"])

X_vif = sm.add_constant(X_vif)

X_vif = X_vif.dropna()

vif = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) 
            for i in range(X_vif.shape[1])]
})

vif = vif.sort_values("VIF", ascending=False)

vif

# Regression Modeling – Claim Cost Prediction

To predict insurance claim amounts, a **Random Forest Regressor** is implemented.

Random Forest is an ensemble learning method that combines multiple decision trees. This approach provides:

• High predictive accuracy  
• Reduced overfitting  
• Ability to model nonlinear relationships

Model performance is evaluated using:

• **R² score**  
• **Root Mean Squared Error (RMSE)**

In [ ]:
# Target
y = df["total_claim_amount"]

# Features
X = df.drop(columns=["total_claim_amount","fraud_reported"])


X = pd.get_dummies(X, drop_first=True)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

# Train
rf.fit(X_train, y_train)
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(10)

# Prediction
pred = rf.predict(X_test)

# Metrics
print("R2:", r2_score(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(8,6))

importance.head(15).plot(kind="barh")

plt.title("Top Feature Importance")
plt.savefig("../reports/figures/feature_importance.png")
plt.show()

# Fraud Detection Model

Detecting fraudulent insurance claims is a high-priority task for insurance companies.

In this step we train a **Random Forest Classifier** to distinguish between legitimate and fraudulent claims.

In [ ]:
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Features and target
X = numeric_df.drop(columns=["fraud_reported"], errors='ignore')
y = numeric_df["fraud_reported"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)

# Predict and report
pred = clf.predict(X_test)
print(classification_report(y_test, pred))

In [ ]:
cm = confusion_matrix(y_test,pred)

sns.heatmap(cm,annot=True,fmt="d")

plt.title("Confusion Matrix")
plt.savefig("../reports/figures/Confusion_Matrix.png")
plt.show()

In [ ]:
prob = clf.predict_proba(X_test)[:,1]

fpr,tpr,_ = roc_curve(y_test,prob)

plt.plot(fpr,tpr)

plt.plot([0,1],[0,1],"--")

plt.title("ROC Curve")
plt.savefig("../reports/figures/roc_curve.png")
plt.show()

# Customer Segmentation using Clustering

Clustering is used to identify natural groupings within the dataset.

This analysis applies **K-Means clustering** to discover segments of policyholders with similar claim characteristics.

In [ ]:
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Features (drop target if present)
features = numeric_df.drop(columns=["fraud_reported"], errors='ignore')

# Scale numeric features
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

# KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(scaled)

In [ ]:
pca = PCA(n_components=2)

pca_res = pca.fit_transform(scaled)

plt.figure(figsize=(10,6))

plt.scatter(
    pca_res[:,0],
    pca_res[:,1],
    c=clusters,
    alpha=0.7
)

plt.title("Customer Segmentation via PCA")

plt.show()

In [ ]:
import pickle

with open("../models/random_forest_model.pkl","wb") as f:
    pickle.dump(rf,f)

# Business Insights

The analysis reveals several key insights:

• Claim costs vary significantly across policyholders  
• Certain demographic and policy characteristics strongly influence claim size  
• Random Forest models provide strong predictive performance  
• Fraud classification models show promising detection capability  

These insights demonstrate how machine learning techniques can support insurance companies in improving risk assessment and fraud detection processes.

# Final Report

## Key Findings

• Claim amount distribution is highly skewed.

• Several demographic and policy variables influence claim cost.

• Random Forest achieved strong performance for claim prediction.

• Fraud detection model shows promising classification accuracy.

## Business Impact

These models can help insurance companies:

- improve risk assessment
- detect fraud earlier
- reduce financial losses
- optimize claim investigation processes